# Gamma Exposure Offline Pipeline Demo

This notebook is the teaching centerpiece for the project. It is **offline-first**: it reads only local Parquet files under `data/raw/` and does not connect to ClickHouse.

We study whether daily options gamma structure is associated with next-day intraday behavior for `SPY`.

In [ ]:
import json
from pathlib import Path

from gamma_exposure_engine.data.raw_store import (
    load_raw_intraday_bars,
    load_raw_options_snapshot,
)
from gamma_exposure_engine.exposure.aggregation import build_daily_gamma_factors
from gamma_exposure_engine.exposure.cleaning import (
    clean_options_snapshot,
    summarize_cleaning_diagnostics,
)
from gamma_exposure_engine.intraday.metrics import (
    attach_pinning_distance,
    build_daily_intraday_metrics,
)
from gamma_exposure_engine.pipeline.offline_pipeline import (
    build_spot_close_frame,
    run_offline_analysis,
    select_pinning_candidates,
)
from gamma_exposure_engine.research.bootstrap import build_quantile_summary_with_ci
from gamma_exposure_engine.research.dataset import build_research_dataset
from gamma_exposure_engine.research.descriptive import (
    build_alternative_band_sensitivity,
    build_leave_one_month_out_sensitivity,
    build_near_spot_share_threshold_summary,
    build_subperiod_stability,
)
from gamma_exposure_engine.research.predictive import (
    build_expanding_window_diagnostics,
    build_predictive_baseline_comparison,
)
from gamma_exposure_engine.research.regime import build_regime_quantile_summary
from gamma_exposure_engine.research.statistical_tests import (
    build_statistical_test_summary,
)

## Step 1 - Read the Offline Raw Data Contract

The canonical local inputs are:

- `data/raw/SPY_intraday_bars.parquet`
- `data/raw/SPY_options_snapshot.parquet`
- `data/raw/manifest.json`

We read `manifest.json` to get the exact date range shipped with the repository.

In [ ]:
current_dir = Path.cwd()
project_root = (
    current_dir if (current_dir / "data" / "raw").exists() else current_dir.parent
)
raw_dir = project_root / "data" / "raw"
manifest_path = raw_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
start_date = manifest["date_range"]["start_date"]
end_date = manifest["date_range"]["end_date"]
symbol = manifest["symbol"]

print(f"Symbol: {symbol}")
print(f"Date range: {start_date} to {end_date}")
print(f"Raw directory: {raw_dir}")

## Step 2 - Load Raw Local Data

We now load intraday bars and options snapshots from local Parquet files only.

Notation used later:

- $t$: an exposure trade date
- $t+1$: the next observed response date

In [ ]:
intraday_bars = load_raw_intraday_bars(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    raw_data_dir=raw_dir,
)
options_snapshot = load_raw_options_snapshot(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    raw_data_dir=raw_dir,
)

print("Intraday rows:", intraday_bars.height)
print("Options rows:", options_snapshot.height)
intraday_bars.head(3), options_snapshot.head(3)

## Step 3 - Build Spot Close and Clean Options

Each option row needs a daily underlying close (`spot_close`) so we can compute moneyness and gamma exposure consistently.

We also run cleaning diagnostics before building factors.

In [ ]:
spot_close = build_spot_close_frame(intraday_bars)
options_with_spot = options_snapshot.join(spot_close, on="trade_date", how="inner")
cleaning_diagnostics = summarize_cleaning_diagnostics(options_with_spot)
cleaned_options = clean_options_snapshot(options_with_spot)

print("Spot close rows:", spot_close.height)
print("Cleaned options rows:", cleaned_options.height)
cleaning_diagnostics

## Step 4 - Build Gamma Factors and Intraday Response Metrics

From cleaned options we build daily gamma factors. From intraday bars we build daily response metrics such as realized variance and abnormal volume score.

We also compute a pinning-distance proxy using prior-day top gamma strikes.

In [ ]:
near_spot_band_width = 0.02
abnormal_volume_window = 20
pinning_candidate_count = 5

gamma_factors = build_daily_gamma_factors(
    cleaned_options,
    near_spot_band=near_spot_band_width,
)
intraday_metrics = build_daily_intraday_metrics(
    intraday_bars,
    abnormal_volume_window=abnormal_volume_window,
)
pinning_candidates = select_pinning_candidates(
    cleaned_options=cleaned_options,
    candidate_count=pinning_candidate_count,
)
intraday_metrics = attach_pinning_distance(intraday_metrics, pinning_candidates)

gamma_factors.head(3), intraday_metrics.head(3)

## Step 5 - Align Exposure Day $t$ to Response Day $t+1$

The core timing contract is no-lookahead alignment:

- exposure factors are measured on day $t$
- response metrics are matched from next observed day $t+1$

This keeps the association analysis temporally honest.

In [ ]:
research_dataset = build_research_dataset(
    exposures=gamma_factors,
    responses=intraday_metrics,
)

print("Aligned research rows:", research_dataset.height)
research_dataset.head(5)

## Step 6 - Descriptive and Inferential Analysis

We now evaluate one factor-target pair.

- factor: `total_open_interest_weighted_gamma`
- target: `next_day_realized_variance`

We compute quantile means with bootstrap confidence intervals, then non-parametric significance tests.

In [ ]:
factor_name = "total_open_interest_weighted_gamma"
target_name = "next_day_realized_variance"
quantiles = 5

quantile_summary = build_quantile_summary_with_ci(
    frame=research_dataset,
    factor_name=factor_name,
    target_name=target_name,
    quantiles=quantiles,
    bootstrap_iterations=500,
    confidence_level=0.95,
)
statistical_tests = build_statistical_test_summary(
    frame=research_dataset,
    factor_name=factor_name,
    target_name=target_name,
    quantiles=quantiles,
)

quantile_summary, statistical_tests

## Step 7 - Regime and Robustness Analysis

Regime analysis checks whether behavior differs in low-volatility versus high-volatility states. Robustness analysis checks sensitivity to thresholds, subperiod splits, alternative near-spot band definitions, and leave-one-month-out exclusions.

In [ ]:
regime_summary = build_regime_quantile_summary(
    frame=research_dataset,
    factor_name=factor_name,
    target_name=target_name,
    quantiles=5,
    lookback_window=20,
)
threshold_summary = build_near_spot_share_threshold_summary(
    frame=research_dataset,
    target_name=target_name,
    thresholds=(0.2, 0.4, 0.6),
)
subperiod_summary = build_subperiod_stability(
    frame=research_dataset,
    factor_name=factor_name,
    target_name=target_name,
)
target_frame = research_dataset.select("trade_date", target_name)
band_sensitivity = build_alternative_band_sensitivity(
    cleaned_options=cleaned_options,
    targets=target_frame,
    target_name=target_name,
    band_widths=(0.01, 0.03, 0.05),
)
loo_month = build_leave_one_month_out_sensitivity(
    frame=research_dataset,
    factor_name=factor_name,
    target_name=target_name,
)

(
    regime_summary.head(5),
    threshold_summary,
    subperiod_summary,
    band_sensitivity,
    loo_month,
)

## Step 8 - Predictive Evaluation

Predictive checks use walk-forward evaluation (expanding training window) to compare:

- linear baseline
- Ridge regression baseline
- naive lagged-target baseline

This is still association-focused and not a trading strategy backtest.

In [ ]:
predictive_comparison = build_predictive_baseline_comparison(
    frame=research_dataset,
    feature_name=factor_name,
    target_name=target_name,
    min_train_size=20,
    ridge_alpha_candidates=(0.01, 0.1, 1.0, 10.0),
)
expanding_window_diagnostics = build_expanding_window_diagnostics(
    frame=research_dataset,
    feature_name=factor_name,
    target_name=target_name,
    min_train_size=20,
    alpha_candidates=(0.01, 0.1, 1.0, 10.0),
)

predictive_comparison, expanding_window_diagnostics.head(5)

## Step 9 - Run the End-to-End Offline CLI Pipeline

To confirm reproducibility, we run the same logic through the offline pipeline entrypoint. It writes non-HTML artifacts (Parquet, CSV, JSON) that can be reused for interviews and blog drafting.

In [ ]:
notebook_output_dir = project_root / "outputs" / "notebook_demo"
run_manifest = run_offline_analysis(
    start_date=start_date,
    end_date=end_date,
    output_dir=notebook_output_dir,
    symbol=symbol,
    raw_data_dir=raw_dir,
)

artifact_rows = {
    name: meta["row_count"] for name, meta in run_manifest["artifacts"].items()
}
print("Output dir:", notebook_output_dir)
print("Artifacts:", sorted(artifact_rows.keys()))
artifact_rows

## Step 10 - Interpretation and Limitations

Interpretation checklist:

- quantify direction and magnitude of quantile differences
- check inferential support (Spearman, Kruskal-Wallis)
- verify regime and robustness stability
- compare predictive baselines without over-claiming

Limitations to state clearly:

- empirical association only (not causal inference)
- sample-window dependence
- single-symbol scope
- sensitivity to options data conventions and cleaning choices

In [ ]:
print("Notebook completed fully offline from local Parquet inputs.")
print("Rows in aligned research dataset:", research_dataset.height)
print("Rows in predictive comparison table:", predictive_comparison.height)